In [ ]:
import MDAnalysis as mda
from MDAnalysis.lib.distances import distance_array

u = mda.Universe("dna_lipids.gro")


dna = u.select_atoms(
    "(resname HFG HSG DG DG3 DG5 and (name O4' C4' C3' C2' C1' N9 C8 N7 C5 C6 O6 N1 C2 N2 N3 C4)) "
    "or (resname HFA HSA DA DA3 DA5 and (name O4' C4' C3' C2' C1' N9 C8 N7 C5 C6 N6 N1 C2 N3 C4)) "
    "or (resname HFC HSC DC DC3 DC5 and (name O4' C4' C3' C2' C1' N1 C2 O2 N3 C4 N4 C5 C6)) "
    "or (resname HFT HST DT DT3 DT5 and (name O4' C4' C3' C2' C1' N1 C2 O2 N3 C4 O4 C5 C5M C6))"
)



lipids = u.select_atoms("resname DMPC DMTAP")

if dna.n_atoms == 0:
    raise ValueError("DNA selection is empty. Check DNA residue names in your GRO.")

clashing_resids = []
dna_pos = dna.positions

for res in lipids.residues:
    if res.atoms.n_atoms == 0:
        continue
    dist = distance_array(res.atoms.positions, dna_pos)
    if dist.size and dist.min() < 2.3:  # Angstrom
        clashing_resids.append(res.resid)

# Exclude only lipid residues that clash
if clashing_resids:
    exclude = "not (resname DMPC DMTAP and resid %s)" % " ".join(map(str, sorted(set(clashing_resids))))
    final_system = u.select_atoms(exclude)
else:
    final_system = u.atoms


print(f"Removed {len(set(clashing_resids))} clashing lipid residues.")
clashing_resids = list(map(int, clashing_resids))
print("Clashing residue IDs:", sorted(clashing_resids))

dmpc_before  = u.select_atoms("resname DMPC").residues.n_residues
dmtap_before = u.select_atoms("resname DMTAP").residues.n_residues
print(f"Number of residues before removal: DMPC={dmpc_before}, DMTAP={dmtap_before}, total={dmpc_before + dmtap_before}")

dmpc_after  = final_system.select_atoms("resname DMPC").residues.n_residues
dmtap_after = final_system.select_atoms("resname DMTAP").residues.n_residues
print(f"Number of residues before removal: DMPC={dmpc_after}, DMTAP={dmtap_after}, total={dmpc_after + dmtap_after}")

final_system.write("out.gro")